In [29]:
import os
import cv2
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

In [53]:
class Image:
    def __init__(self, impath: str, 
                 metric: str = 'ncc', 
                 use_pyramid: bool = False, 
                 auto_contrast: bool = True, 
                 auto_white_balance: bool = True, 
                 auto_color_correction: bool = True
                ):
        self.im = cv2.imread(impath)
        self.use_pyramid = use_pyramid
        self.auto_contrast = auto_contrast
        self.auto_white_balance = auto_white_balance
        self.auto_color_correction = auto_color_correction

        # convert to double
        self.img = self.im.astype(np.float32) / 255.0

        # compute the height of each part (1/3 of total)
        self.shape = self.img.shape
        self.height = np.floor(self.shape[0] / 3.0).astype(int)

        # align the channels
        self.img_out = self.align_channels(metric, use_pyramid)

    def align_channels(self, metric, use_pyramid):
        aligner = Alignment(self, metric=metric, use_pyramid=use_pyramid)

        enhancer = ImageEnhancer(aligner.img_out)
        enhanced_img_out = enhancer.enhance(
            auto_contrast=self.auto_contrast, 
            auto_white_balance=self.auto_white_balance, 
            auto_color_correction=self.auto_color_correction,
        )

        print(f"R offset: {aligner.displacements['r']}")
        print(f"G offset: {aligner.displacements['g']}")
        print(f"B offset: {aligner.displacements['b']}")
        return enhanced_img_out
    
    def show_image(self):
        # display the image
        plt.imshow(self.img_out)
        plt.axis('off')
        plt.show()
    
    def save_image(self, fname):
        # save the image
        os.makedirs('out', exist_ok=True)
        img_to_save = (self.img_out * 255).astype(np.uint8)
        img_bgr = cv2.cvtColor(img_to_save, cv2.COLOR_RGB2BGR)
        cv2.imwrite(fname, img_bgr)
        

class Alignment:
    def __init__(self, img: Image, metric='ncc', use_pyramid=False):
        # parameters
        self.refine_range = 3
        self.max_shift = 50
        self.max_levels = 6
        self.min_size = 50
        
        self.img = img
        self.metric = metric
        self.use_pyramid = use_pyramid

        self.displacements = defaultdict(lambda: (0, 0))
        r, g, b = self.get_channels()
        r_aligned, g_aligned, b_aligned = self.align(r, g, b, ref='g')
        self.img_out = self.combine_channels(r_aligned, g_aligned, b_aligned)
        self.img_out = self.crop_borders(self.img_out)

    def get_channels(self):
        # separate color channels
        b = self.img.img[:self.img.height, :, 0]
        g = self.img.img[self.img.height: 2*self.img.height, :, 0]
        r = self.img.img[2*self.img.height: 3*self.img.height, :, 0]
        return r, g, b
    
    def combine_channels(self, r, g, b):
        # create a color image
        img_out = np.dstack([r, g, b])
        return img_out
    
    def get_edges(self, img):
        # convert image to edge features
        dx = cv2.Sobel(img, cv2.CV_64F, 1, 0, ksize=3)
        dy = cv2.Sobel(img, cv2.CV_64F, 0, 1, ksize=3)
        magnitude = np.sqrt(dx**2 + dy**2)
        return magnitude
    
    def align(self, r, g, b, ref='g'):
        if ref == 'b':
            r_aligned = self.align_channel(b, r, 'r', self.use_pyramid)
            g_aligned = self.align_channel(b, g, 'g', self.use_pyramid)
            return r_aligned, g_aligned, b
        if ref == 'g':
            r_aligned = self.align_channel(g, r, 'r', self.use_pyramid)
            b_aligned = self.align_channel(g, b, 'b', self.use_pyramid)
            return r_aligned, g, b_aligned
        if ref == 'r':
            g_aligned = self.align_channel(r, g, 'g', self.use_pyramid)
            b_aligned = self.align_channel(r, b, 'b', self.use_pyramid)
            return r, g_aligned, b_aligned
    
    def simple_align(self, ref, target, search_range=None, use_edges=False):
        if search_range is None:
            search_range = self.max_shift

        if use_edges:
            ref = self.get_edges(ref)
            target = self.get_edges(target)

        # higher ncc is better, lower l2norm is better
        best_metric = -np.inf if self.metric == 'ncc' else np.inf
        best_dx, best_dy = 0, 0
                
        ref_cropped = self.crop_borders(ref)
        
        # iterate over all shifts in the search range
        for dy in range(-search_range, search_range + 1):
            for dx in range(-search_range, search_range + 1):
                # translate image
                aligned = self.translate(target, dx, dy)
                # crop borders to avoid edge effects
                aligned_cropped = self.crop_borders(aligned)
                
                if self.metric == 'ncc':
                    score = self.ncc(ref_cropped, aligned_cropped)
                    if score > best_metric:
                        best_metric = score
                        best_dx, best_dy = dx, dy
                else:  # L2 norm
                    score = self.l2norm(ref_cropped, aligned_cropped)
                    if score < best_metric:
                        best_metric = score
                        best_dx, best_dy = dx, dy
        return best_dx, best_dy
    
    def align_channel(self, ref, target, channel_name='', use_pyramid=False):
        if use_pyramid:
            best_dx, best_dy = self.pyramid_search(ref, target)
        else:
            best_dx, best_dy = self.simple_align(ref, target)
    
        # check alignment quality
        aligned = self.translate(target, best_dx, best_dy)
        ref_cropped = self.crop_borders(ref)
        aligned_cropped = self.crop_borders(aligned)
        score = self.ncc(ref_cropped, aligned_cropped)
        
        # retry with edges if alignment is poor
        if score < 0.7:
            if use_pyramid:
                best_dx, best_dy = self.pyramid_search(ref, target, use_edges=True)
            else:
                best_dx, best_dy = self.simple_align(ref, target, use_edges=True)
        
        if channel_name:
            self.displacements[channel_name] = (best_dx, best_dy)
        
        return self.translate(target, best_dx, best_dy)
    
    def pyramid_search(self, ref, target, level=0, max_levels=None, use_edges=False):
        # calculate max levels based on image size
        if max_levels is None:
            min_dim = min(ref.shape)
            # stop when image smaller than min_size pixels
            max_levels = int(np.log2(min_dim / self.min_size))
            max_levels = max(1, min(max_levels, self.max_levels))
        
        # at coarsest level, do exhaustive search
        if level >= max_levels:
            return self.simple_align(ref, target, search_range=self.max_shift, use_edges=use_edges)
        
        # downsample both images by factor of 2
        ref_small = self.downsample(ref)
        target_small = self.downsample(target)
        
        # recursive call to get coarse estimate
        dx_coarse, dy_coarse = self.pyramid_search(
            ref_small, target_small, level + 1, max_levels, use_edges=use_edges
        )
        
        # scale up the estimate
        dx = dx_coarse * 2
        dy = dy_coarse * 2
        
        # align image with coarse estimate then
        # refine with small local search at current resolution
        target_coarse_aligned = self.translate(target, dx, dy)
        
        # search in small window around the estimate
        dx_refine, dy_refine = self.simple_align(
            ref, target_coarse_aligned, search_range=self.refine_range, use_edges=use_edges
        )
        
        # combine coarse and refined estimates
        return dx + dx_refine, dy + dy_refine
    
    def downsample(self, img, factor=2):
        h, w = img.shape[:2]
        new_size = (w // factor, h // factor)
        return cv2.resize(img, new_size, interpolation=cv2.INTER_AREA)
    
    def crop_borders(self, img, crop_percent=0.1):
        h, w = img.shape[:2]
        crop_h = int(h * crop_percent)
        crop_w = int(w * crop_percent)
        return img[crop_h:-crop_h, crop_w:-crop_w]
    
    def translate(self, img, dx, dy):
        # translate image by (dx, dy)
        return np.roll(img, shift=(dy, dx), axis=(0, 1))
    
    def l2norm(self, original, aligned):
        diff = original - aligned
        return np.sqrt(np.sum(diff ** 2))
    
    def ncc(self, original, aligned):
        original = original - np.mean(original)
        aligned = aligned - np.mean(aligned)
        original_mag = np.sqrt(np.sum(original ** 2))
        aligned_mag = np.sqrt(np.sum(aligned ** 2))
        if original_mag == 0 or aligned_mag == 0:
            return -1
        return np.sum(original * aligned) / (original_mag * aligned_mag)
    

class ImageEnhancer:
    
    def __init__(self, img):
        self.img = img
    
    def enhance(self, auto_contrast=True, auto_white_balance=True, auto_color_correction=True):

        result = self.img.copy()

        # auto contrast
        if auto_contrast:
            result = self.auto_contrast(result)
        
        # white balance
        if auto_white_balance:
            result = self.auto_white_balance(result)
        
        # color correction
        if auto_color_correction:
            result = self.auto_color_correction(result)
        
        return result
    

    # bells and whistles
    
    def auto_contrast(self, img, percentile=1):
        # automatically adjust contrast by stretching intensity range.
        img_contrast = np.zeros_like(img)
        
        for c in range(img.shape[2] if len(img.shape) == 3 else 1):
            if len(img.shape) == 3:
                channel = img[:, :, c]
            else:
                channel = img
            
            # avoid outliers
            p_low = np.percentile(channel, percentile)
            p_high = np.percentile(channel, 100 - percentile)
            
            # clip and normalize
            channel_clipped = np.clip(channel, p_low, p_high)
            
            if p_high > p_low:
                channel_normalized = (channel_clipped - p_low) / (p_high - p_low)
            else:
                channel_normalized = channel_clipped
            
            if len(img.shape) == 3:
                img_contrast[:, :, c] = channel_normalized
            else:
                img_contrast = channel_normalized
        
        return img_contrast
    
    def auto_white_balance(self, img, percentile=95):
        # automatic white balancing assuming brightest pixels are white

        # get brightest pixels from each channel
        bright_r = np.percentile(img[:, :, 0], percentile)
        bright_g = np.percentile(img[:, :, 1], percentile)
        bright_b = np.percentile(img[:, :, 2], percentile)
        
        # find max brightness
        max_bright = max(bright_r, bright_g, bright_b)
        
        # scale each channel
        img_balanced = np.zeros_like(img)
        img_balanced[:, :, 0] = np.clip(img[:, :, 0] * (max_bright / bright_r), 0, 1)
        img_balanced[:, :, 1] = np.clip(img[:, :, 1] * (max_bright / bright_g), 0, 1)
        img_balanced[:, :, 2] = np.clip(img[:, :, 2] * (max_bright / bright_b), 0, 1)
        
        return img_balanced
    
    def auto_color_correction(self, img):
        # adaptive color correction based on image statistics

        # calculate channel statistics
        mean_vals = np.mean(img, axis=(0, 1))
        std_vals = np.std(img, axis=(0, 1))
        
        # normalize each channel to have similar statistics
        img_corrected = np.zeros_like(img)
        target_mean = np.mean(mean_vals)
        target_std = np.mean(std_vals)
        
        for c in range(3):
            img_corrected[:, :, c] = (img[:, :, c] - mean_vals[c]) * (target_std / std_vals[c]) + target_mean
        
        return np.clip(img_corrected, 0, 1)

In [ ]:
image_names_small = ['cathedral.jpg', 'monastery.jpg', 'tobolsk.jpg']
for imname in image_names_small:
    print(imname)
    impath = os.path.join('data', imname)
    image = Image(impath, metric='ncc', use_pyramid=False)
    fname = os.path.join('out', f'simple_aligned_{imname.split("/")[-1]}')
    image.save_image(fname)
    image.show_image()

In [ ]:
image_names = os.listdir('data')
for imname in tqdm(image_names):
    if not imname.lower().endswith(('.png', '.jpg', '.jpeg', '.tif')):
        continue
    print(imname)
    impath = os.path.join('data', imname)
    image = Image(impath, use_pyramid=True)
    fname = os.path.join('out', f'pyramid_aligned_{imname.split("/")[-1].split(".")[0]}.jpg')
    image.save_image(fname)
    image.show_image()

In [ ]:
image_names = os.listdir('data')
for imname in tqdm(image_names):
    if not imname.lower().endswith(('.png', '.jpg', '.jpeg', '.tif')):
        continue
    impath = os.path.join('data', imname)
    image = Image(impath, use_pyramid=True, auto_contrast=False, auto_white_balance=False, auto_color_correction=False)
    fname = os.path.join('out', f'pyramid_aligned_noenhancement_{imname.split("/")[-1].split(".")[0]}.jpg')
    image.save_image(fname)
    image.show_image()

In [ ]:
image_names = os.listdir('data')
for imname in tqdm(image_names):
    if not imname.lower().endswith(('.png', '.jpg', '.jpeg', '.tif')):
        continue
    impath = os.path.join('data', imname)
    image = Image(impath, use_pyramid=True, auto_contrast=True, auto_white_balance=False, auto_color_correction=False)
    fname = os.path.join('out', f'pyramid_aligned_autocontrast_{imname.split("/")[-1].split(".")[0]}.jpg')
    image.save_image(fname)
    image.show_image()

In [ ]:
image_names = os.listdir('data')
for imname in tqdm(image_names):
    if not imname.lower().endswith(('.png', '.jpg', '.jpeg', '.tif')):
        continue
    impath = os.path.join('data', imname)
    image = Image(impath, use_pyramid=True, auto_contrast=False, auto_white_balance=True, auto_color_correction=False)
    fname = os.path.join('out', f'pyramid_aligned_autowhitebalance_{imname.split("/")[-1].split(".")[0]}.jpg')
    image.save_image(fname)
    image.show_image()

In [ ]:
image_names = os.listdir('data')
for imname in tqdm(image_names):
    if not imname.lower().endswith(('.png', '.jpg', '.jpeg', '.tif')):
        continue
    impath = os.path.join('data', imname)
    image = Image(impath, use_pyramid=True, auto_contrast=False, auto_white_balance=False, auto_color_correction=True)
    fname = os.path.join('out', f'pyramid_aligned_autocolorcorrection_{imname.split("/")[-1].split(".")[0]}.jpg')
    image.save_image(fname)
    image.show_image()